same code written as a jupyter notebook file 
#exactly similar code is written without brief explanations in another file that I have submitted '24K-0826-runner.py' which is a python  file this has this entire code written collectively since my jypyter notebook was trucating down the output for some cells which caused issues for me in taking screenshots for my report so I copied all the python code cells from here to there, so that I can run the code in my terminal and get the full non trucated version of the output, Initially my approach was to complete the assignment on a jupyter notebook file

Name: Muhammad Abu Bakar
Roll Number: 24K-0826
Assignment: Kaggle Playground Series ML Assignment 1

For this assignment, I was to predict irrigation needs (Low, Medium, High). Since the dataset has over 600,000 rows, memory and speed are big issues. I am using a Jupyter Notebook approach to break down my steps, test the required models, and find the best one without crashing my computer.

In [17]:

import numpy as np
import pandas as pd
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import RidgeClassifier
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, ClassifierMixin


RNG_SEED = 42

First, I am loading the training and testing datasets. I'm printing out the shapes and the class distribution to see what I have at hand. The data is super imbalanced as the 'High' irrigation class is being rare.

In [18]:
print("Loading data from disk.")

train_raw = pd.read_csv("train.csv")
test_raw  = pd.read_csv("test.csv")

test_ids = test_raw["id"].values

print(f"Train shape : {train_raw.shape}")
print(f"Test shape  : {test_raw.shape}")
print(f"Class distribution:\n{train_raw['Irrigation_Need'].value_counts()}\n")

Loading data from disk.
Train shape : (630000, 21)
Test shape  : (270000, 20)
Class distribution:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64



Next, I am splitting the data into features (the inputs) and the target (what I am supposed to get my code to predict).  also converted the  labels (Low, Medium, High) into numbers so the models can read them.

In [19]:
DROP_COLS    = ["id", "Irrigation_Need"]
TARGET_COL   = "Irrigation_Need"


categorical_cols = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage",
    "Season", "Irrigation_Type", "Water_Source",
    "Mulching_Used", "Region"
]
numerical_cols = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon",
    "Electrical_Conductivity", "Temperature_C", "Humidity",
    "Rainfall_mm", "Sunlight_Hours", "Wind_Speed_kmh",
    "Field_Area_hectare", "Previous_Irrigation_mm"
]

feature_cols   = categorical_cols + numerical_cols
feature_matrix = train_raw[feature_cols].copy()
target_labels  = train_raw[TARGET_COL].copy()
test_features  = test_raw[feature_cols].copy()


label_enc        = LabelEncoder()
target_encoded   = label_enc.fit_transform(target_labels)

print(f"Encoded classes : {list(label_enc.classes_)}")
print(f"Label mapping   : {dict(zip(label_enc.classes_, label_enc.transform(label_enc.classes_)))}\n")

Encoded classes : ['High', 'Low', 'Medium']
Label mapping   : {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}



To handle the text columns, 
I am using an OrdinalEncoder instead of OneHotEncoder. 
OneHotEncoder would create way too many new columns and itni RAM mere pass hai nahi, so imo OrdinalEncoder is a much safer and better choice for 630k rows.

In [20]:

categorical_transformer = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)



soil_crop_pipeline = ColumnTransformer(
    transformers=[
        ("cat_ord", categorical_transformer, categorical_cols),
        ("num_pass", "passthrough",            numerical_cols),
    ],
    remainder="drop",
    n_jobs=1
)

As of what Sir usama taught us K-Means is usually for grouping unlabeled data, not classifying things. To make it work, I wrote a quick custom class. It groups the data into 3 clusters, checks which label is most common in each cluster, and just guesses that label.

In [21]:
class KMeansClassifierWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, n_clusters=3, random_state=RNG_SEED, n_init=10, max_iter=300):
        self.n_clusters   = n_clusters
        self.random_state = random_state
        self.n_init       = n_init
        self.max_iter     = max_iter

    def fit(self, feature_block, label_block):
       

        self._km = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=self.n_init,
            max_iter=self.max_iter
        )
        cluster_ids = self._km.fit_predict(feature_block)

        self._cluster_label_map = {}
        for cluster_id in range(self.n_clusters):
            mask = cluster_ids == cluster_id
            if mask.sum() == 0:
                self._cluster_label_map[cluster_id] = 0
            else:
                counts = np.bincount(label_block[mask].astype(int),
                                     minlength=len(np.unique(label_block)))
                self._cluster_label_map[cluster_id] = int(np.argmax(counts))

        self.classes_ = np.unique(label_block)
        return self

    def predict(self, feature_block):
        cluster_ids = self._km.predict(feature_block)
        return np.array([self._cluster_label_map[c] for c in cluster_ids])

For testing purposes;
I can't use Leave-One-Out Cross Validation (LOOCV). 
Running a test 630,000 separate times would take hours on my laptop. 
Instead, I am using 5-Fold Stratified Cross Validation. This splits the data into 5 chunks but ensures that the rare 'High' class is spread evenly so the models get a fair chance to learn it.

In [22]:
def run_stratified_cv(model_pipeline, feature_matrix, target_encoded, n_splits=5, label="Model"):
    skf         = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RNG_SEED)
    fold_scores = []
    last_cm     = None
    t0          = time.perf_counter()

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(feature_matrix, target_encoded), start=1):
        fold_train_features = feature_matrix.iloc[train_idx]
        fold_val_features   = feature_matrix.iloc[val_idx]
        fold_train_labels   = target_encoded[train_idx]
        fold_val_labels     = target_encoded[val_idx]

        model_pipeline.fit(fold_train_features, fold_train_labels)
        fold_predictions = model_pipeline.predict(fold_val_features)

        fold_ba = balanced_accuracy_score(fold_val_labels, fold_predictions)
        fold_scores.append(fold_ba)
        last_cm = confusion_matrix(fold_val_labels, fold_predictions)

        print(f"    Fold {fold_idx}/5 — balanced_accuracy = {fold_ba:.4f}")

    elapsed      = time.perf_counter() - t0
    mean_bal_acc = float(np.mean(fold_scores))
    std_bal_acc  = float(np.std(fold_scores))

    print(f"  ── {label} CV Summary ──")
    print(f"     Mean Balanced Accuracy : {mean_bal_acc:.4f}")
    print(f"     Total time             : {elapsed:.1f}s")
    print(f"\n  Confusion Matrix (last fold)")
    print(f"  Classes: {list(label_enc.classes_)}")
    print(f"{last_cm}\n")

    return mean_bal_acc, last_cm

Phase 1: Testing the base models. I am testing the four algorithms the assignment asked for. Naive Bayes, Ridge, and K-Means 

all of these will  struggle here because they don't handle the complicated relationships between soil, rain, and crops [as given in outr data set] very well. The Decision Tree easily wins and gives us the most favorable res.

In [23]:
print("PHASE 1 — Base Model Benchmarks\n")

# 1. Decision Tree 
tree_model = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", DecisionTreeClassifier(max_depth=10, random_state=RNG_SEED))
])
dt_base_score, _ = run_stratified_cv(tree_model, feature_matrix, target_encoded, label="Decision Tree (base)")

# 2. Naive Bayes
nb_pipeline = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", GaussianNB())
])
nb_score, _ = run_stratified_cv(nb_pipeline, feature_matrix, target_encoded, label="Gaussian Naive Bayes")

# 3. Ridge Classifier, 
# Using linear regression for classification
# I tried messing with the alpha parameter but it didn't help much

ridge_pipeline = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", RidgeClassifier(alpha=1.0, class_weight="balanced"))
])
ridge_score, _ = run_stratified_cv(ridge_pipeline, feature_matrix, target_encoded, label="Ridge Classifier")

# 4. K-Means
kmeans_pipeline = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", KMeansClassifierWrapper(n_clusters=3, random_state=RNG_SEED))
])
kmeans_score, _ = run_stratified_cv(kmeans_pipeline, feature_matrix, target_encoded, label="K-Means Wrapper")

PHASE 1 — Base Model Benchmarks

    Fold 1/5 — balanced_accuracy = 0.9618
    Fold 2/5 — balanced_accuracy = 0.9621
    Fold 3/5 — balanced_accuracy = 0.9617
    Fold 4/5 — balanced_accuracy = 0.9608
    Fold 5/5 — balanced_accuracy = 0.9609
  ── Decision Tree (base) CV Summary ──
     Mean Balanced Accuracy : 0.9615
     Total time             : 45.3s

  Confusion Matrix (last fold)
  Classes: ['High', 'Low', 'Medium']
[[ 3836     0   366]
 [    0 73661   322]
 [  175  1054 46586]]

    Fold 1/5 — balanced_accuracy = 0.6900
    Fold 2/5 — balanced_accuracy = 0.6906
    Fold 3/5 — balanced_accuracy = 0.6891
    Fold 4/5 — balanced_accuracy = 0.6869
    Fold 5/5 — balanced_accuracy = 0.6897
  ── Gaussian Naive Bayes CV Summary ──
     Mean Balanced Accuracy : 0.6893
     Total time             : 8.2s

  Confusion Matrix (last fold)
  Classes: ['High', 'Low', 'Medium']
[[ 1768    30  2404]
 [   10 66692  7281]
 [  744 11351 35720]]

    Fold 1/5 — balanced_accuracy = 0.6814
    Fold 2/5

Phase 2: Tuning the Decision Tree. Now that I know the tree is the best, 
I will move on towards trying a few different ways to tune it.

1) First, I let it grow with no limits, which caused it to overfit. 
2)Then, I limited the depth but didn't balance the classes, so it ignored the rare 'High' needs. 
3) Finally, I found the best combination: a depth of 20 with balanced class weights so it pays attention to everything fairly.

In [24]:
print("PHASE 2 — Decision Tree Tuning\n")

# Attempt 1 - Overfitting the tree - FAILED I AM SED....
# The tree just memorizes the training data and fails on anything new.
overfitted_tree = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", DecisionTreeClassifier(max_depth=None, random_state=RNG_SEED))
])
overfitted_score, overfitted_cm = run_stratified_cv(overfitted_tree, feature_matrix, target_encoded, label="Overfitted Decision Tree")

# Attempt 2 - Ignored the class imbalance - FAILED I AM SED AGAIN.....
# Without telling it to care about the minority class, it gets lazy and guesses 'Low' too often. IT IS AS LAZY AS ME tch tch...
imbalanced_tree = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", DecisionTreeClassifier(max_depth=15, random_state=RNG_SEED))
])
imbalanced_score, imbalanced_cm = run_stratified_cv(imbalanced_tree, feature_matrix, target_encoded, label="Imbalanced Decision Tree")

# Winner Hai: Optimized Tree
# Balanced weights force the tree to treat the 'HIGHH' minority class like it's important.
optimised_tree = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", DecisionTreeClassifier(
        max_depth=20,
        class_weight="balanced",
        min_samples_leaf=10,    
        random_state=RNG_SEED
    ))
])
optimised_score, optimised_cm = run_stratified_cv(optimised_tree, feature_matrix, target_encoded, label="Optimised Decision Tree")

PHASE 2 — Decision Tree Tuning

    Fold 1/5 — balanced_accuracy = 0.9456
    Fold 2/5 — balanced_accuracy = 0.9481
    Fold 3/5 — balanced_accuracy = 0.9475
    Fold 4/5 — balanced_accuracy = 0.9447
    Fold 5/5 — balanced_accuracy = 0.9453
  ── Overfitted Decision Tree CV Summary ──
     Mean Balanced Accuracy : 0.9463
     Total time             : 85.0s

  Confusion Matrix (last fold)
  Classes: ['High', 'Low', 'Medium']
[[ 3761     1   440]
 [    0 72449  1534]
 [  506  1325 45984]]

    Fold 1/5 — balanced_accuracy = 0.9557
    Fold 2/5 — balanced_accuracy = 0.9586
    Fold 3/5 — balanced_accuracy = 0.9591
    Fold 4/5 — balanced_accuracy = 0.9562
    Fold 5/5 — balanced_accuracy = 0.9575
  ── Imbalanced Decision Tree CV Summary ──
     Mean Balanced Accuracy : 0.9574
     Total time             : 63.8s

  Confusion Matrix (last fold)
  Classes: ['High', 'Low', 'Medium']
[[ 3819     0   383]
 [    0 73477   506]
 [  298  1116 46401]]

    Fold 1/5 — balanced_accuracy = 0.9588
    

Phase 3: The final step. 
I take the optimized Decision Tree and train it one last time on the entire training dataset. 
Then I make the predictions for the test set and save it to a CSV file to upload to Kaggle.

In [25]:
print("PHASE 3 — Final Training & Submission\n")

final_pipeline = Pipeline([
    ("preproc",    soil_crop_pipeline),
    ("classifier", DecisionTreeClassifier(
        max_depth=20,
        class_weight="balanced",
        min_samples_leaf=10,
        random_state=RNG_SEED
    ))
])

print("Fitting the final model on all the training data...")
final_pipeline.fit(feature_matrix, target_encoded)

print("Making predictions for test.csv...")
test_predictions_encoded = final_pipeline.predict(test_features)
test_predictions_labels  = label_enc.inverse_transform(test_predictions_encoded)

# Creating the final submission file
submission_df = pd.DataFrame({
    "id":              test_ids,
    "Irrigation_Need": test_predictions_labels
})
submission_path = "final_submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Saved: {submission_path}")
print("Ssab kuch Khatam, FINALLY!!!!!")

PHASE 3 — Final Training & Submission

Fitting the final model on all the training data...
Making predictions for test.csv...
Saved: final_submission.csv
Ssab kuch Khatam, FINALLY!!!!!
